# Sistema de Recomendación con CatBoost - Ejemplo con Datos Propios

Este notebook muestra cómo usar el sistema de recomendación con tus propios datos.

In [ ]:
import pandas as pd
import numpy as np
from src.model import RecommendationModel
from src.preprocessing import DataPreprocessor
from src.evaluation import Evaluator

# Configuración del modelo
model = RecommendationModel(
    learning_rate=0.1,
    iterations=1000,
    # Configura los nombres de las columnas según tus datos
    user_id_col='tu_columna_usuario',
    item_id_col='tu_columna_item',
    relevance_col='tu_columna_relevancia',
    # Si tienes características categóricas, especifica aquí
    categorical_features=['caracteristica_categorica1', 'caracteristica_categorica2']
)

In [ ]:
# Cargar tus datos
# Ajusta las rutas según dónde estén tus archivos
interactions_df = pd.read_csv('data/tus_interacciones.csv')
user_features_df = pd.read_csv('data/tus_caracteristicas_usuarios.csv')  # Opcional
item_features_df = pd.read_csv('data/tus_caracteristicas_items.csv')    # Opcional

# Muestra las primeras filas de los datos
display(interactions_df.head())
display(user_features_df.head()) if user_features_df is not None else None
display(item_features_df.head()) if item_features_df is not None else None

In [ ]:
# Preprocesamiento de datos
preprocessor = DataPreprocessor()
processed_df = preprocessor.preprocess_interactions(interactions_df)

# Dividir en train y test
train_df, test_df, train_group_id, test_group_id = 
    preprocessor.prepare_training_data(processed_df)

# Mostrar tamaño de los conjuntos
print(f'Tamaño del conjunto de entrenamiento: {len(train_df)}')
print(f'Tamaño del conjunto de test: {len(test_df)}')

In [ ]:
# Preparar datos para el modelo
X_train, y_train = model.prepare_data(
    train_df,
    user_features_df,
    item_features_df
)

# Entrenar el modelo
model.train(X_train, y_train, train_group_id)

In [ ]:
# Evaluar el modelo
evaluator = Evaluator()
ndcg_score = evaluator.evaluate_recommender(model, test_df)
print(f'NDCG Score: {ndcg_score:.4f}')

In [ ]:
# Hacer recomendaciones para un usuario específico
# Cambia este ID por el ID del usuario que quieras analizar
user_id = 'tu_id_usuario'

# Obtener todos los items disponibles como candidatos
candidates_df = pd.DataFrame({
    model.user_id_col: [user_id],
    model.item_id_col: item_features_df[model.item_id_col]  # Si tienes características de items
})

# Obtener recomendaciones
recommendations = model.recommend(
    user_id,
    candidates_df,
    user_features_df,
    item_features_df,
    top_n=10  # Número de recomendaciones a generar
)
print(f'Recomendaciones para el usuario {user_id}:')
recommendations